# Build the Yorùbá tone listening PILOT (thin wrapper, run tonight)

This notebook is a **thin wrapper** around `pilot/build_pilot_form.py` + `pilot/score_pilot.py` (all the
real logic lives there, grounded in nb14's native loader and nb07's `tone_i2`). It:

1. installs deps + clones this repo (for the `pilot/` scripts),
2. runs the build → `pilot/pilot_form.html` (self-contained, mobile-first) + `pilot/keymap.json` (withheld),
3. **plays the 4 ear-check clips inline** so you confirm the audio is sane before you send the form,
4. (optional) scores a reviewer's pasted answers.

**GPU:** a T4/L4 is enough (native download + scoring); the model clips synthesize with OmniVoice. If
OmniVoice/the 5h checkpoint is missing the build DEGRADES to a native+catch form (it still ships). **Send the
single `pilot_form.html` to your native reviewer over WhatsApp/email — it works offline.**

## 1. Install + restart (numpy<2, scoring stack, omnivoice, tone-metric, ffmpeg)

In [ ]:
%pip install --quiet "numpy<2"
%pip install --quiet omnivoice
%pip install --quiet boto3 soundfile soxr librosa speechbrain "swift-f0" pyworld praat-parselmouth transformers safetensors "huggingface_hub>=0.24.0" tqdm
%pip install --quiet --no-deps "git+https://github.com/mosesdaudu001/tone-on-a-budget.git"
%pip uninstall -y hf-xet
import subprocess; subprocess.run(["apt-get","-qq","install","-y","ffmpeg"], check=False)
import os
print("Installs done. RESTARTING so numpy<2 takes effect — run from the NEXT cell.", flush=True)
os._exit(0)

In [ ]:
import numpy as np
assert np.__version__.startswith("1."), "numpy<2 pin did not take — re-run install + restart"
import os, shutil, subprocess
# clone this repo to get the pilot/ scripts (build_pilot_form.py + score_pilot.py)
REPO = "/content/tone-on-a-budget" if os.path.isdir("/content") else "/tmp/tone-on-a-budget"
if os.path.isdir(REPO): shutil.rmtree(REPO)
subprocess.run(["git","clone","--depth","1","https://github.com/mosesdaudu001/tone-on-a-budget.git",REPO], check=True)
PILOT = os.path.join(REPO, "pilot")
assert os.path.exists(os.path.join(PILOT,"build_pilot_form.py")), PILOT
print("numpy", np.__version__, "OK |", PILOT)

## 2. Secrets → env (so the build script's `_secret` finds them without prompting)

In [ ]:
import os, getpass
def _secret(k):
    try:
        from google.colab import userdata
        v = userdata.get(k)
        if v: return v
    except Exception: pass
    return os.environ.get(k) or getpass.getpass(f"{k}: ")
os.environ["AWS_ACCESS_KEY_ID"] = _secret("AWS_ACCESS_KEY_ID")
os.environ["AWS_SECRET_ACCESS_KEY"] = _secret("AWS_SECRET_ACCESS_KEY")
os.environ["AWS_DEFAULT_REGION"] = os.environ.get("AWS_DEFAULT_REGION", "us-east-1")
_hf = _secret("HF_TOKEN"); os.environ["HF_TOKEN"] = _hf or ""
if _hf:
    from huggingface_hub import login; login(token=_hf)
os.environ["HF_HUB_DISABLE_XET"] = "1"; os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "0"
print("secrets in env ✓ (AWS + HF)")

## 3. (optional) DRY RUN — list the clips it WILL use, download/synthesize nothing

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, os.path.join(PILOT,"build_pilot_form.py"),
                "--dry-run", "--n-native","12","--n-model0h","9","--n-model5h","9","--n-catch","6"],
               check=True, env=os.environ.copy())

## 4. BUILD the form (downloads native, synthesizes model0h/5h, scores tone_i2, embeds audio)

Writes `pilot/pilot_form.html` + `pilot/keymap.json` **inside the cloned repo**. ~36 items, a few MB.

In [ ]:
import subprocess, sys
OUT = PILOT  # write next to the scripts
subprocess.run([sys.executable, os.path.join(PILOT,"build_pilot_form.py"),
                "--n-native","12","--n-model0h","9","--n-model5h","9","--n-catch","6",
                "--seed","4242","--catch-mode","dup","--out-dir",OUT],
               check=True, env=os.environ.copy())
HTML = os.path.join(OUT,"pilot_form.html"); KEYMAP = os.path.join(OUT,"keymap.json")
print("\nbuilt:", HTML, "(%.2f MB)" % (os.path.getsize(HTML)/1e6))

## 5. EAR-CHECK — play the 4 ear-check clips inline (decode the embedded audio)

Decisive sanity check before you send: do the clips play, and does each one's written text match what you
hear? (You are not judging tone here — just that the audio + text pairing is intact.)

In [ ]:
import json, re, base64, tempfile, IPython.display as ipd
html = open(HTML, encoding="utf-8").read()
ITEMS = json.loads(re.search(r"const ITEMS = (\[.*?\]);", html, re.S).group(1))
KEY = json.load(open(KEYMAP, encoding="utf-8"))
tmp = tempfile.mkdtemp()
for it in ITEMS[:4]:                                     # the 4 ear-check ids printed by the build
    iid = it["item_id"]; km = KEY[iid]
    header, b64 = it["audio"].split(",", 1)
    ext = "mp3" if "mpeg" in header else "wav"
    fp = f"{tmp}/{iid}.{ext}"; open(fp,"wb").write(base64.b64decode(b64))
    src = km["source"]; ti2 = km["tone_i2"]
    print(f"{iid} | source={src} | tone_i2={ti2} | text: {it['text']}")
    ipd.display(ipd.Audio(filename=fp))

## 6. Download the form to send (Colab)

Grab `pilot_form.html` and send it to your native reviewer. `keymap.json` is the **withheld key — never send
it.** (`pilot/1_PING_message.md` + `pilot/2_REVIEWER_instructions.md` are the messages to send alongside.)

In [ ]:
try:
    from google.colab import files; files.download(HTML)
except Exception:
    print("(not on Colab) the form is at:", HTML)

## 7. (optional) SCORE a reviewer's pasted answers

Paste the reviewer's `PILOT1;item01=ok;...` code into `ANSWERS` below and run. Repeat per rater.

In [ ]:
import subprocess, sys
ANSWERS = "PILOT1;item01=ok;item02=bad;item03=unsure"   # <- paste the reviewer's pasted code here
subprocess.run([sys.executable, os.path.join(PILOT,"score_pilot.py"),
                "--keymap", KEYMAP, "--answers", ANSWERS], check=True, env=os.environ.copy())